# Buy-only Monthly Investing with an ML Tilt — End-to-End Workflow

This notebook calls the functions in `src/` and shows the full workflow:
**download → transform → predict → simulate → this month's buy list**.

All logic lives in the `.py` files; this notebook only orchestrates and visualises.

In [ ]:
# In Google Colab: clone the repo first, then install dependencies
# !git clone <YOUR_REPO_URL> && %cd <REPO_FOLDER>
# !pip install -r requirements.txt -q

import sys, os
sys.path.append(os.path.abspath(".."))       # notebook lives in notebooks/; repo root holds config.py
if not os.path.exists("config.py") and os.path.exists("../config.py"):
    os.chdir("..")                           # run from the repo root so data/ and outputs/ resolve

import pandas as pd
import matplotlib.pyplot as plt
import config as cfg
from src.data_loader import load_data
from src.features import build_monthly_table
from src.modeling import add_split, tune_models, fit_final_and_predict, test_report, feature_importance
from src.simulation import compare_strategies, latest_buy_list
pd.set_option("display.width", 160)

## Step 1 — Data
* `SOURCE = "download"`: fetch fresh prices (yfinance) and macro data (FRED, VIX), then cache them in `data/`
* `SOURCE = "file"`: reuse the cached files (fast, no internet)

In [ ]:
SOURCE = "download"      # first run: "download"; afterwards "file"
prices, macro = load_data(SOURCE, cfg)
prices.groupby("Ticker")["Date"].agg(["min", "max", "count"])

## Step 2 — Features and the monthly table
One row per asset and month (first trading day = buying day). Feature sets are defined explicitly.

In [ ]:
monthly, fs = build_monthly_table(prices, macro, cfg)
monthly = add_split(monthly, cfg.TRAIN_END, cfg.VALID_END)
FEATURES = fs["FEATURES"]
for k, v in fs.items():
    print(f"{k:<11} ({len(v)}): {v}")
monthly.groupby("split")["ym"].agg(["min", "max", "count"])

## Step 3 — Exploratory analysis
* How often does an asset beat the All-World benchmark over 6 months?
* Which features correlate with the target?

In [ ]:
TARGET = "is_outperform_6m"
labeled = monthly[monthly[TARGET].notna()]
print("Share of 'outperform' by split:")
print(labeled.groupby("split")[TARGET].mean().round(3), "\n")
print("Share of 'outperform' by ticker:")
print(labeled.groupby("Ticker")[TARGET].mean().sort_values(ascending=False).round(3))

In [ ]:
corr = labeled[fs["NUMERICAL"] + [TARGET]].corr()[TARGET].drop(TARGET)
corr.reindex(corr.abs().sort_values(ascending=False).index).round(3).to_frame("corr_with_target")

## Step 4 — Modelling
Decision Tree and Random Forest, `max_depth` tuned on VALIDATION (precision).
A 6-month embargo before each split prevents label overlap (leakage).

In [ ]:
tuning = tune_models(monthly, FEATURES, cfg)
tuning.head(10)

In [ ]:
best = tuning.dropna(subset=["val_precision"]).iloc[0]
model, monthly = fit_final_and_predict(monthly, FEATURES, cfg, best["model"], int(best["max_depth"]))
print(f"Final model: {best['model']} (max_depth={int(best['max_depth'])})\n")
print(test_report(monthly).round(3).to_string(index=False))
print("\nTop features:\n", feature_importance(model, FEATURES).round(3).to_string())

**Reading the test report:** `n_months` = number of distinct months with a signal.
All assets are compared with the same benchmark, so signals concentrated in a few
months can look precise by luck. Precision should be judged together with `n_months`.

## Step 5 — Buy-only simulation (TEST period, out-of-sample)
| Strategy | Monthly contribution |
|---|---|
| A | 100% All-World (passive benchmark) |
| B | 25% each core ETF |
| C0 | 70% core + 30% split equally over all satellite stocks (no ML) |
| C | 70% core + 30% to the top-3 satellite stocks by model probability |
| D | C, but contribution doubled when All-World is ≥5% below its high |

D invests more money in total, so compare strategies by **IRR**, not by final value.

In [ ]:
summary, curves = compare_strategies(monthly, cfg, period_mask=monthly["split"] == "test")
summary.round(4)

In [ ]:
ax = curves.plot(figsize=(11, 5), title="Portfolio value — buy-only monthly investing (TEST period)")
ax.set_xlabel("month"); ax.set_ylabel("portfolio value"); plt.show()

## Step 6 — This month's buy list
The latest month has no label yet (its 6-month future is unknown), but the model can still score it.

In [ ]:
latest_buy_list(monthly, cfg).round(4)